In [1]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [2]:
tokenizer.is_fast

True

In [3]:
import os
from datasets import load_dataset

dataset_name = "SQuAD_2.0"

train_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "train-00000-of-00001.parquet")
val_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "validation-00000-of-00001.parquet")

dataset = load_dataset("parquet", data_files={'train': train_path, 'val': val_path})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [10]:
from datasets import DatasetDict

# time to reduce the dataset
reduced_train_set = dataset["train"].shuffle(seed=42).select(range(6500))
reduced_val_set = dataset["val"].shuffle(seed=42).select(range(650))
reduced_test_set = dataset["val"].shuffle(seed=42).select(range(650,1300))

reduced_set = DatasetDict({"train":reduced_train_set,
                           "val":reduced_val_set,
                           "test":reduced_test_set})

reduced_set

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 6500
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 650
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 650
    })
})

In [55]:
stride = 128
max_length = 384

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    # inputs contain tokenised questions and context in same list, separated by [SEP] token
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,              # max number of tokens for each input
        truncation="only_second",           # only truncates the second thing which is the context
        stride = stride,                    # controls the number of tokens that overlap
        return_overflowing_tokens=True,     # maps each input to a question in case of long contexts
        return_offsets_mapping=True,        # keeps track of which character index each token begins and ends at
        padding="max_length",               # pads each input to the max length
    )
    # [CLS] token at the beginning of each input, [SEP] to separate question and context

    offset_mapping = inputs["offset_mapping"]
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
    example_ids = []

    for i, offset in enumerate(offset_mapping):
        # accounting for any overlapping
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        example_ids.append(examples["id"][sample_idx])
        # accounts for questions with no answer and use CLS token
        if len(answer["answer_start"]) == 0:
            start_char = 0
            end_char = 0
        else:
            start_char = answer["answer_start"][0]
            end_char = answer["answer_start"][0] + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        offset_mapping[i] = [
            (o if s == 1 else None) for o, s in zip(offset, sequence_ids)
        ]

        # Find the start and end of the context
        # At the question, sequence id = 0, for context sequence id = 1
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    inputs["example_id"] = example_ids
    return inputs

In [56]:
tokenized_dataset = {}
for split in reduced_set:
    tokenized_dataset[split] = reduced_set[split].map(
        preprocess_function,
        batched=True,
        remove_columns=reduced_set[split].column_names,
    )
# tokenized_dataset.set_format("torch")
# tokenized_dataset = DatasetDict(tokenized_dataset).set_format("torch")
print(f"{len(reduced_set["train"]), len(tokenized_dataset["train"])}")
print(f"{len(reduced_set["val"]), len(tokenized_dataset["val"])}")
print(f"{len(reduced_set["test"]), len(tokenized_dataset["test"])}")
tokenized_dataset

Map:   0%|          | 0/6500 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

(6500, 6570)
(650, 663)
(650, 659)


{'train': Dataset({
     features: ['input_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'example_id'],
     num_rows: 6570
 }),
 'val': Dataset({
     features: ['input_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'example_id'],
     num_rows: 663
 }),
 'test': Dataset({
     features: ['input_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'example_id'],
     num_rows: 659
 })}

In [46]:
from transformers import AutoModelForQuestionAnswering
import torch

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

model = model.to(device)

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [72]:
from tqdm.auto import tqdm
import numpy as np
import collections
import evaluate
import torch.nn.functional as F

n_best = 20
max_answer_length = 30

metric = evaluate.load("squad_v2")
print(metric)

def compute_metrics(start_logits, end_logits, features, examples):
    """
    Arguments:
        start_logits: start token scores
        end_logits: end token scores
        features: tokenized inputs
        examples: non tokenized data
    """
    # group truncated contexts back together, example_to_feature is a dictionary with a list of indexes
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            no_answer_logit = start_logit[0] + end_logit[0]
            no_answer_prob = F.softmax(torch.tensor(no_answer_logit), dim=-1)

            # sorts the logits in ascending order, getting the indicies for the top nbest scores
            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                        "no_answer_prob": no_answer_prob.to(torch.float32),
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            no_answer = max(answers, key=lambda x:x["no_answer_prob"])
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": no_answer["no_answer_prob"].item()}
            )
        else:
            # if no answer, then no answer probability is 1
            predicted_answers.append({"id": example_id, "prediction_text": "", "no_answer_probability": 1.0})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers, no_answer_threshold = 0.5)

EvaluationModule(name: "squad_v2", module_type: "metric", features: {'predictions': {'id': Value(dtype='string', id=None), 'prediction_text': Value(dtype='string', id=None), 'no_answer_probability': Value(dtype='float32', id=None)}, 'references': {'id': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}}, usage: """
Computes SQuAD v2 scores (F1 and EM).
Args:
    predictions: List of triple for question-answers to score with the following elements:
        - the question-answer 'id' field as given in the references (see below)
        - the text of the answer
        - the probability that the question has no answer
    references: List of question-answers dictionaries with the following key-values:
            - 'id': id of the question-answer pair (see above),
            - 'answers': a list of Dict {'text': text of the answer as a string}
    no_answer_threshold: fl

In [47]:
batch = {k: torch.tensor(tokenized_dataset["val"][k]).to(device) for k in tokenized_dataset["val"].column_names}

with torch.no_grad():
    outputs = model(**batch)

In [48]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()

In [73]:
results = compute_metrics(start_logits, end_logits, tokenized_dataset["val"], reduced_set["val"])
results

  0%|          | 0/650 [00:00<?, ?it/s]

{'exact': 51.23076923076923,
 'f1': 51.23076923076923,
 'total': 650,
 'HasAns_exact': 0.0,
 'HasAns_f1': 0.0,
 'HasAns_total': 317,
 'NoAns_exact': 100.0,
 'NoAns_f1': 100.0,
 'NoAns_total': 333,
 'best_exact': 51.23076923076923,
 'best_exact_thresh': 0.0,
 'best_f1': 51.23076923076923,
 'best_f1_thresh': 0.0}